# Accounts overview

Lists every account in the active data repo with its opening balance and (if applicable) the most recent rate that the scenario applies to it.

`DATA_DIR` comes from `$ANALYSIS_DATA_DIR`. The scenario defaults the same way the `sim` CLI resolves it: `$ANALYSIS_SCENARIO` if set, else the name recorded in the data repo's `.current_scenario` (written by `sim load <scenario>`), else `current` — use the `Scenario:` dropdown below to pick a different one, then re-run this cell and the ones below.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import Markdown, display, clear_output
from ipywidgets import widgets
import pandas as pd

from rich.console import Console
from analysis.sim.accounts import (
    load_accounts_view, render_accounts_table, accounts_dataframe,
)

from analysis.sim.loader import resolve_scenario_name
from analysis.sim.notebook_ui.pickers import scenario_picker, dollar_plot_mode
from analysis.sim.notebook_ui.accounts_plots import (
    account_history_widget, 
)

DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", "../data/shared/sim"))
# Same resolution the CLI uses: $ANALYSIS_SCENARIO, else the data repo's
# .current_scenario, else "current".
_default_scenario = resolve_scenario_name(
    DATA_DIR, os.environ.get("ANALYSIS_SCENARIO") or None
).name
AS_OF = None  # e.g. '2026-05-31' to pin rate lookups
cumulative_caveat =  "_Note that expense and income flows are cumulative totals._  \n"

PLOT_MODE = dollar_plot_mode()

# Pick a different scenario here, then re-run this cell and the ones below.
# CONNECTIONS holds the live ScenarioConnection/RunConnection pair; `run` is
# None if this scenario hasn't been run yet (fine for the first two sections
# below -- only "Balance history" needs a run).
PICKER, CONNECTIONS = scenario_picker(DATA_DIR, _default_scenario, require_run=False)
PICKER

## Rich table

In [ ]:
def show_rows(_, scenario):
    rows = load_accounts_view(scenario, as_of=AS_OF)
    Console().print(render_accounts_table(rows, title=f"accounts ({DATA_DIR.name})"))
CONNECTIONS.show(show_rows, scenario=CONNECTIONS.scenario)

## DataFrame view

Same data as a pandas DataFrame for filtering, grouping, or export.

In [ ]:
df = None
def set_df(**_):
    global df
    df = accounts_dataframe(CONNECTIONS.scenario, as_of=AS_OF)
CONNECTIONS.observe(set_df, names="value")
df = accounts_dataframe(CONNECTIONS.scenario, as_of=AS_OF)
def show_dataframe(_, df: pd.DataFrame):
    clear_output()
    display(df)

CONNECTIONS.show(show_dataframe, df=df)

In [ ]:
# Summary: total opening balance per kind.
def _summary_by_kind(_, df: pd.DataFrame):
    if not df.empty:
        summary = (
            df.dropna(subset=["opening_balance"])
            .groupby("kind", as_index=False)["opening_balance"]
            .sum()
            .sort_values("opening_balance", ascending=False)
        )
        clear_output()
        display(summary)
CONNECTIONS.show(_summary_by_kind, df=df)

## Balance history

Pick one or more accounts (Cmd/Ctrl-click for multi-select) to plot their
month-end balances from the scenario's most recent run (the `latest run`
shown above).

Assets/liabilities and income/expense are plotted separately below since
they're on very different scales and mean different things.


In [ ]:
def show_run_info(_, scenario):
    if CONNECTIONS.run is None:
        raise FileNotFoundError(
            f"no runs for {CONNECTIONS.scenario_ref.name!r} in "
            f"{CONNECTIONS.scenario_ref.runs_dir}; run `sim run`"
        )
    rc = CONNECTIONS.run
    RUN_FILE = rc.run_path
    print(f"{len(rc.account_kinds())} accounts, {len(rc.balance_pivot())} months")

CONNECTIONS.show(show_run_info, scenario=CONNECTIONS.scenario)

In [ ]:
CONNECTIONS.show(account_history_widget, "asset", "Asset balances",
                plot_mode=PLOT_MODE)

### Liability history


In [ ]:
CONNECTIONS.show(account_history_widget, "liability", "Liability balances",
                plot_mode=PLOT_MODE)

### Income history

> *NOTE: Income values here are cummulative totals.*


In [ ]:
CONNECTIONS.show(account_history_widget, "income", "Income balances",
                plot_mode=PLOT_MODE,
                caveat=cumulative_caveat)


### Expense history

> *NOTE: Expense values here are cummulative totals.*

In [ ]:
CONNECTIONS.show(account_history_widget, "expense", "Expense balances",
                plot_mode=PLOT_MODE,
                caveat=cumulative_caveat)
